In [1]:
import numpy as np

In [2]:
def combine_multiple_datasets(file_list, out_file, verbose=True):
    """
    Combine multiple direct-BC datasets with variable trajectory lengths into one dataset.

    Expected saved fields in each input file:
        qs      : (n_traj, T, dof)
        xb      : (n_traj, T, n_b)
        idx_b   : (n_b,) or (n_traj, n_b)
        lambdas : (T,) or (n_traj, T)

    Output saved fields:
        qs      : (N_total, T_max, dof)
        xb      : (N_total, T_max, n_b)
        idx_b   : (n_b,)   if all files share same idx_b
                 or (N_total, n_b) if per-trajectory idx_b is needed
        lambdas : (N_total, T_max)
        valid   : (N_total, T_max)

    Padding strategy:
        - prepend padding in time dimension
        - qs, xb padded with edge values
        - lambdas padded with 0
        - valid is False on padded entries, True on original entries
    """
    if len(file_list) == 0:
        raise ValueError("file_list must contain at least one file.")

    datasets = [np.load(f) for f in file_list]

    # ----------------------------
    # Helpers
    # ----------------------------
    def normalize_lambdas(data_dict):
        """Return lambdas as shape (n_traj, T)."""
        qs = data_dict["qs"]
        lam = data_dict["lambdas"]
        n_traj = qs.shape[0]

        if lam.ndim == 1:
            # shared lambda vector -> repeat for each trajectory in this file
            lam = np.broadcast_to(lam[None, :], (n_traj, lam.shape[0]))
        elif lam.ndim == 2:
            if lam.shape[0] != n_traj:
                raise ValueError(
                    f"lambdas has shape {lam.shape}, but qs has {n_traj} trajectories."
                )
        else:
            raise ValueError(
                f"lambdas must have ndim 1 or 2, got shape {lam.shape}."
            )

        return lam

    def normalize_idx_b(data_dict):
        """Return idx_b as shape (n_traj, n_b)."""
        qs = data_dict["qs"]
        idx_b = data_dict["idx_b"]
        n_traj = qs.shape[0]

        if idx_b.ndim == 1:
            idx_b = np.broadcast_to(idx_b[None, :], (n_traj, idx_b.shape[0]))
        elif idx_b.ndim == 2:
            if idx_b.shape[0] != n_traj:
                raise ValueError(
                    f"idx_b has shape {idx_b.shape}, but qs has {n_traj} trajectories."
                )
        else:
            raise ValueError(
                f"idx_b must have ndim 1 or 2, got shape {idx_b.shape}."
            )

        return idx_b

    def process_data(data_dict, target_l):
        """
        Normalize and pad one dataset to target_l.
        Returns:
            qs_pad      : (n_traj, target_l, dof)
            xb_pad      : (n_traj, target_l, n_b)
            lam_pad     : (n_traj, target_l)
            valid       : (n_traj, target_l)
            idx_b_all   : (n_traj, n_b)
        """
        qs = data_dict["qs"]
        xb = data_dict["xb"]
        lam = normalize_lambdas(data_dict)
        idx_b_all = normalize_idx_b(data_dict)

        n_traj, current_l, dof = qs.shape
        _, xb_l, n_b = xb.shape

        if xb_l != current_l:
            raise ValueError(
                f"Inconsistent time dimension: qs has T={current_l}, xb has T={xb_l}."
            )
        if lam.shape[1] != current_l:
            raise ValueError(
                f"Inconsistent time dimension: qs has T={current_l}, lambdas has T={lam.shape[1]}."
            )

        diff = target_l - current_l
        if diff < 0:
            raise ValueError(
                f"target_l={target_l} is smaller than current_l={current_l}."
            )

        qs_pad = np.pad(qs, ((0, 0), (diff, 0), (0, 0)), mode="edge")
        xb_pad = np.pad(xb, ((0, 0), (diff, 0), (0, 0)), mode="edge")
        lam_pad = np.pad(lam, ((0, 0), (diff, 0)), mode="constant", constant_values=0)

        valid = np.concatenate(
            [
                np.zeros((n_traj, diff), dtype=bool),
                np.ones((n_traj, current_l), dtype=bool),
            ],
            axis=1,
        )

        return qs_pad, xb_pad, lam_pad, valid, idx_b_all

    # ----------------------------
    # Global checks
    # ----------------------------
    lengths = []
    dof_ref = None
    nb_ref = None

    for k, d in enumerate(datasets):
        qs = d["qs"]
        xb = d["xb"]

        if qs.ndim != 3:
            raise ValueError(f"{file_list[k]}: qs must have shape (n_traj, T, dof), got {qs.shape}")
        if xb.ndim != 3:
            raise ValueError(f"{file_list[k]}: xb must have shape (n_traj, T, n_b), got {xb.shape}")

        dof = qs.shape[2]
        n_b = xb.shape[2]
        T = qs.shape[1]

        if dof_ref is None:
            dof_ref = dof
        elif dof != dof_ref:
            raise ValueError(
                f"{file_list[k]}: dof mismatch. Expected {dof_ref}, got {dof}."
            )

        if nb_ref is None:
            nb_ref = n_b
        elif n_b != nb_ref:
            raise ValueError(
                f"{file_list[k]}: boundary size mismatch. Expected {nb_ref}, got {n_b}."
            )

        lengths.append(T)

    max_l = max(lengths)

    # ----------------------------
    # Process and collect
    # ----------------------------
    qs_blocks = []
    xb_blocks = []
    lam_blocks = []
    valid_blocks = []
    idx_blocks = []

    for file_name, d in zip(file_list, datasets):
        qs_i, xb_i, lam_i, valid_i, idx_i = process_data(d, max_l)

        qs_blocks.append(qs_i)
        xb_blocks.append(xb_i)
        lam_blocks.append(lam_i)
        valid_blocks.append(valid_i)
        idx_blocks.append(idx_i)

        if verbose:
            print(f"\nProcessed {file_name}")
            print("  qs      ", qs_i.shape)
            print("  xb      ", xb_i.shape)
            print("  lambdas ", lam_i.shape)
            print("  valid   ", valid_i.shape)
            print("  idx_b   ", idx_i.shape)

    # ----------------------------
    # Concatenate across trajectories
    # ----------------------------
    qs = np.concatenate(qs_blocks, axis=0)
    xb = np.concatenate(xb_blocks, axis=0)
    lambdas = np.concatenate(lam_blocks, axis=0)
    valid = np.concatenate(valid_blocks, axis=0)
    idx_b_all = np.concatenate(idx_blocks, axis=0)

    # If all trajectories share same idx_b, save compact 1D version.
    if np.all(idx_b_all == idx_b_all[0]):
        idx_b_out = idx_b_all[0]
    else:
        idx_b_out = idx_b_all

    if verbose:
        print("\nFINAL")
        print("  qs      ", qs.shape)
        print("  xb      ", xb.shape)
        print("  lambdas ", lambdas.shape)
        print("  valid   ", valid.shape)
        print("  idx_b   ", idx_b_out.shape)

    np.savez(
        out_file,
        qs=qs,
        xb=xb,
        idx_b=idx_b_out,
        lambdas=lambdas,
        valid=valid,
    )

    if verbose:
        print(f"\nCombined {len(file_list)} files into {out_file}.")

### Slinky 3 noded: 
/slinky_final_data/3_noded/

In [3]:
slinky_3_noded_location = "slinky_final_data/3_noded/"

# training data
combine_multiple_datasets(
    [
        f"{slinky_3_noded_location}n3_traj1_dataset_52_pts.npz",
        # f"{slinky_3_noded_location}n3_traj2_dataset_12_pts.npz", init condition is not = reference shape
        f"{slinky_3_noded_location}n3_traj3_dataset_12_pts.npz",
        f"{slinky_3_noded_location}n3_traj4_dataset_12_pts.npz",
    ],
    "n3_slinky_train_dataset.npz",
)

# test data
combine_multiple_datasets(
    [
        f"{slinky_3_noded_location}n3_traj5_dataset_22_pts.npz",
        f"{slinky_3_noded_location}n3_traj6_dataset_22_pts.npz",
        f"{slinky_3_noded_location}n3_traj7_dataset_55_pts.npz",
        f"{slinky_3_noded_location}n3_traj8_dataset_55_pts.npz",
    ],
    "n3_slinky_test_dataset.npz",
)



Processed slinky_final_data/3_noded/n3_traj1_dataset_52_pts.npz
  qs       (1, 51, 11)
  xb       (1, 51, 8)
  lambdas  (1, 51)
  valid    (1, 51)
  idx_b    (1, 8)

Processed slinky_final_data/3_noded/n3_traj3_dataset_12_pts.npz
  qs       (1, 51, 11)
  xb       (1, 51, 8)
  lambdas  (1, 51)
  valid    (1, 51)
  idx_b    (1, 8)

Processed slinky_final_data/3_noded/n3_traj4_dataset_12_pts.npz
  qs       (1, 51, 11)
  xb       (1, 51, 8)
  lambdas  (1, 51)
  valid    (1, 51)
  idx_b    (1, 8)

FINAL
  qs       (3, 51, 11)
  xb       (3, 51, 8)
  lambdas  (3, 51)
  valid    (3, 51)
  idx_b    (8,)

Combined 3 files into n3_slinky_train_dataset.npz.

Processed slinky_final_data/3_noded/n3_traj5_dataset_22_pts.npz
  qs       (1, 54, 11)
  xb       (1, 54, 8)
  lambdas  (1, 54)
  valid    (1, 54)
  idx_b    (1, 8)

Processed slinky_final_data/3_noded/n3_traj6_dataset_22_pts.npz
  qs       (1, 54, 11)
  xb       (1, 54, 8)
  lambdas  (1, 54)
  valid    (1, 54)
  idx_b    (1, 8)

Processed s

### Slinky 5 noded: 
/slinky_final_data/5_noded/

In [4]:
slinky_5_noded_location = "slinky_final_data/5_noded/"
# training data
combine_multiple_datasets(
    [
        f"{slinky_5_noded_location}n5_traj1_dataset_52_pts.npz",
        # f"{slinky_5_noded_location}n5_traj2_dataset_12_pts.npz", init condition is not = reference shape
        f"{slinky_5_noded_location}n5_traj3_dataset_12_pts.npz",
        f"{slinky_5_noded_location}n5_traj4_dataset_12_pts.npz",
    ],
    "n5_slinky_train_dataset.npz",
)

# test data
combine_multiple_datasets(
    [
        f"{slinky_5_noded_location}n5_traj5_dataset_22_pts.npz",
        f"{slinky_5_noded_location}n5_traj6_dataset_22_pts.npz",
        f"{slinky_5_noded_location}n5_traj7_dataset_55_pts.npz",
        f"{slinky_5_noded_location}n5_traj8_dataset_55_pts.npz",
    ],
    "n5_slinky_test_dataset.npz",
)



Processed slinky_final_data/5_noded/n5_traj1_dataset_52_pts.npz
  qs       (1, 51, 27)
  xb       (1, 51, 18)
  lambdas  (1, 51)
  valid    (1, 51)
  idx_b    (1, 18)

Processed slinky_final_data/5_noded/n5_traj3_dataset_12_pts.npz
  qs       (1, 51, 27)
  xb       (1, 51, 18)
  lambdas  (1, 51)
  valid    (1, 51)
  idx_b    (1, 18)

Processed slinky_final_data/5_noded/n5_traj4_dataset_12_pts.npz
  qs       (1, 51, 27)
  xb       (1, 51, 18)
  lambdas  (1, 51)
  valid    (1, 51)
  idx_b    (1, 18)

FINAL
  qs       (3, 51, 27)
  xb       (3, 51, 18)
  lambdas  (3, 51)
  valid    (3, 51)
  idx_b    (18,)

Combined 3 files into n5_slinky_train_dataset.npz.

Processed slinky_final_data/5_noded/n5_traj5_dataset_22_pts.npz
  qs       (1, 54, 27)
  xb       (1, 54, 18)
  lambdas  (1, 54)
  valid    (1, 54)
  idx_b    (1, 18)

Processed slinky_final_data/5_noded/n5_traj6_dataset_22_pts.npz
  qs       (1, 54, 27)
  xb       (1, 54, 18)
  lambdas  (1, 54)
  valid    (1, 54)
  idx_b    (1, 18)


### Strip 9 noded: 
/strip_data/9_noded/

In [5]:
strip_location = "strip_data/9_noded/"

combine_multiple_datasets(
    [
        f"{strip_location}n7_traj1_strip_dataset_19_pts.npz",
        # f"{strip_location}n7_traj2_strip_dataset_14_pts.npz",
        # f"{strip_location}n7_traj3_strip_dataset_14_pts.npz",
        f"{strip_location}n7_traj4_strip_dataset_28_pts.npz",
        f"{strip_location}n7_traj6_strip_dataset_21_pts.npz",
    ],
    "n9_strip_train_dataset.npz",
)

combine_multiple_datasets(
[       f"{strip_location}n7_traj2_strip_dataset_14_pts.npz",
        f"{strip_location}n7_traj3_strip_dataset_14_pts.npz",
        # f"{strip_location}n7_traj4_strip_dataset_28_pts.npz",
        
        f"{strip_location}n7_traj5_strip_dataset_28_pts.npz",
        # f"{strip_location}n7_traj6_strip_dataset_21_pts.npz",
        f"{strip_location}n7_traj7_strip_dataset_21_pts.npz",
        # f"{strip_location} n7_traj8_strip_dataset_21_pts.npz",
    ],
    "n9_strip_test_dataset.npz",
)



Processed strip_data/9_noded/n7_traj1_strip_dataset_19_pts.npz
  qs       (1, 28, 35)
  xb       (1, 28, 20)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 20)

Processed strip_data/9_noded/n7_traj4_strip_dataset_28_pts.npz
  qs       (1, 28, 35)
  xb       (1, 28, 20)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 20)

Processed strip_data/9_noded/n7_traj6_strip_dataset_21_pts.npz
  qs       (1, 28, 35)
  xb       (1, 28, 20)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 20)

FINAL
  qs       (3, 28, 35)
  xb       (3, 28, 20)
  lambdas  (3, 28)
  valid    (3, 28)
  idx_b    (20,)

Combined 3 files into n9_strip_train_dataset.npz.

Processed strip_data/9_noded/n7_traj2_strip_dataset_14_pts.npz
  qs       (1, 28, 35)
  xb       (1, 28, 20)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 20)

Processed strip_data/9_noded/n7_traj3_strip_dataset_14_pts.npz
  qs       (1, 28, 35)
  xb       (1, 28, 20)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 20)

Proce

### Tube 7 noded: 
/tube_data/7_noded/

In [6]:
tube_location = "tube_data/7_noded/"

combine_multiple_datasets(
    [
        f"{tube_location}n7_traj1_tube_dataset_20_pts.npz",
        # f"{tube_location}n7_traj2_tube_dataset_14_pts.npz",
        # f"{tube_location}n7_traj3_tube_dataset_14_pts.npz",
        f"{tube_location}n7_traj4_tube_dataset_28_pts.npz",
        f"{tube_location}n7_traj6_tube_dataset_21_pts.npz",
    ],
    "n7_tube_train_dataset.npz",
)

combine_multiple_datasets(
[       f"{tube_location}n7_traj2_tube_dataset_14_pts.npz",
        f"{tube_location}n7_traj3_tube_dataset_14_pts.npz",
        # f"{tube_location}n7_traj4_tube_dataset_28_pts.npz",
        
        f"{tube_location}n7_traj5_tube_dataset_28_pts.npz",
        # f"{tube_location}n7_traj6_tube_dataset_21_pts.npz",
        f"{tube_location}n7_traj7_tube_dataset_21_pts.npz",
        f"{tube_location}n7_traj8_tube_dataset_27_pts.npz",
    ],
    "n7_tube_test_dataset.npz",
)


Processed tube_data/7_noded/n7_traj1_tube_dataset_20_pts.npz
  qs       (1, 28, 27)
  xb       (1, 28, 12)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 12)

Processed tube_data/7_noded/n7_traj4_tube_dataset_28_pts.npz
  qs       (1, 28, 27)
  xb       (1, 28, 12)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 12)

Processed tube_data/7_noded/n7_traj6_tube_dataset_21_pts.npz
  qs       (1, 28, 27)
  xb       (1, 28, 12)
  lambdas  (1, 28)
  valid    (1, 28)
  idx_b    (1, 12)

FINAL
  qs       (3, 28, 27)
  xb       (3, 28, 12)
  lambdas  (3, 28)
  valid    (3, 28)
  idx_b    (12,)

Combined 3 files into n7_tube_train_dataset.npz.

Processed tube_data/7_noded/n7_traj2_tube_dataset_14_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processed tube_data/7_noded/n7_traj3_tube_dataset_14_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processed tube_d

### Tape 7 noded:

In [7]:
tape_location = "tape_data/7_noded/"

combine_multiple_datasets( # odd trajectories
    [
        f"{tape_location}n7_traj1_tape_dataset_19_pts.npz",
        # f"{tape_location}n7_traj2_tape_dataset_13_pts.npz",
        f"{tape_location}n7_traj3_tape_dataset_13_pts.npz",
        # f"{tape_location}n7_traj4_tape_dataset_27_pts.npz",
        f"{tape_location}n7_traj5_tape_dataset_27_pts.npz",
        # f"{tape_location}n7_traj6_tape_dataset_20_pts.npz",
        f"{tape_location}n7_traj7_tape_dataset_20_pts.npz",
        # f"{tape_location}n7_traj8_tape_dataset_21_pts.npz",
        f"{tape_location}n7_traj9_tape_dataset_16_pts.npz",
        # f"{tape_location}n7_traj10_tape_dataset_16_pts.npz",
        f"{tape_location}n7_traj11_tape_dataset_30_pts.npz",
        # f"{tape_location}n7_traj12_tape_dataset_30_pts.npz",
        f"{tape_location}n7_traj13_tape_dataset_23_pts.npz",
        # f"{tape_location}n7_traj14_tape_dataset_23_pts.npz",
    ],
    "n7_tape_train_dataset.npz",
)

combine_multiple_datasets( # even trajectories
[       
        # f"{tape_location}n7_traj1_tape_dataset_19_pts.npz",
        f"{tape_location}n7_traj2_tape_dataset_13_pts.npz",
        # f"{tape_location}n7_traj3_tape_dataset_13_pts.npz",
        f"{tape_location}n7_traj4_tape_dataset_27_pts.npz",
        # f"{tape_location}n7_traj5_tape_dataset_27_pts.npz",
        f"{tape_location}n7_traj6_tape_dataset_20_pts.npz",
        # f"{tape_location}n7_traj7_tape_dataset_20_pts.npz",
        f"{tape_location}n7_traj8_tape_dataset_21_pts.npz",
        # f"{tape_location}n7_traj9_tape_dataset_16_pts.npz",
        f"{tape_location}n7_traj10_tape_dataset_16_pts.npz",
        # f"{tape_location}n7_traj11_tape_dataset_30_pts.npz",
        f"{tape_location}n7_traj12_tape_dataset_30_pts.npz",
        # f"{tape_location}n7_traj13_tape_dataset_23_pts.npz",
        f"{tape_location}n7_traj14_tape_dataset_23_pts.npz",
    ],
    "n7_tape_test_dataset.npz",
)


Processed tape_data/7_noded/n7_traj1_tape_dataset_19_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processed tape_data/7_noded/n7_traj3_tape_dataset_13_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processed tape_data/7_noded/n7_traj5_tape_dataset_27_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processed tape_data/7_noded/n7_traj7_tape_dataset_20_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processed tape_data/7_noded/n7_traj9_tape_dataset_16_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processed tape_data/7_noded/n7_traj11_tape_dataset_30_pts.npz
  qs       (1, 27, 27)
  xb       (1, 27, 12)
  lambdas  (1, 27)
  valid    (1, 27)
  idx_b    (1, 12)

Processe